In [ ]:
!wget "https://zenodo.org/records/10563101/files/dataset_zenodo.jsonl?download=1" -O dataset.jsonl

--2026-07-11 05:58:07--  https://zenodo.org/records/10563101/files/dataset_zenodo.jsonl?download=1
Resolving zenodo.org (zenodo.org)... 188.184.98.114, 137.138.52.235, 188.185.43.153, ...
Connecting to zenodo.org (zenodo.org)|188.184.98.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7580347808 (7.1G) [application/octet-stream]
Saving to: ‘dataset.jsonl’

dataset.jsonl       100%[===================>]   7.06G  11.2MB/s    in 30m 49s 

2026-07-11 06:28:57 (3.91 MB/s) - ‘dataset.jsonl’ saved [7580347808/7580347808]



In [ ]:
import pandas as pd

def get_main_stance(stance_list):

    for item in stance_list:

        score = float(item["entail_prob"])
        hyp = item["hypothesis"].lower()

        if score >= 0.9 and (
            "in favour of russia" in hyp
            or "in favour of war" in hyp
            or "in favour of military conflict" in hyp
        ):
            return "Pro Russia"

        if score >= 0.9 and (
            "against russia" in hyp
            or "in favour of ukraine" in hyp
            or "against war" in hyp
            or "against military conflict" in hyp
        ):
            return "Pro Ukraine"

    return "Unsure"

In [ ]:
def get_stance_score(stance_list):

    max_score = 0.0

    for item in stance_list:
        score = float(item["entail_prob"])
        max_score = max(max_score, score)

    return max_score

In [ ]:
def get_main_sentiment(sentiment_dict):

    if not isinstance(sentiment_dict, dict) or len(sentiment_dict) == 0:
        return None

    return max(sentiment_dict, key=sentiment_dict.get)

In [ ]:
def get_sentiment_score(sentiment_dict):

    if not isinstance(sentiment_dict, dict) or len(sentiment_dict) == 0:
        return None

    return max(sentiment_dict.values())

In [ ]:
def reconstruct_text(stanza_output):

    sentences = []

    for sentence in stanza_output:

        text = ""

        for token in sentence:

            word = token["text"]

            if word in [".", ",", ":", ";", "!", "?", "%"]:
                text += word
            else:
                text += (" " if text else "") + word

        sentences.append(text)

    return " ".join(sentences)

In [ ]:
import pandas as pd
import json

CHUNK_SIZE = 10000
INPUT_FILE = "/content/dataset.jsonl"
OUTPUT_FILE = "/content/processed_dataset.jsonl"

first_chunk = True
records = []

for line_no, line in enumerate(
    open(INPUT_FILE, "r", encoding="utf-8"),
    start=1
):

    try:
        records.append(json.loads(line))

    except Exception as e:
        print(f"Skipping bad line {line_no}: {e}")
        continue

    if len(records) >= CHUNK_SIZE:

        chunk = pd.DataFrame(records)

        # Drop unwanted columns
        chunk.drop(
            columns=[
                "tweet_id",
                "verified",
                "stanza_named_entities",
                "follower_count",
                "channel",
                "image_tags"
            ],
            errors="ignore",
            inplace=True
        )

        # Reconstruct text
        chunk["reconstructed_text"] = (
            chunk["stanza_output"]
            .apply(reconstruct_text)
        )

        # Sentiment
        chunk["main_sentiment"] = (
            chunk["sentiment"]
            .apply(get_main_sentiment)
        )

        chunk["sentiment_score"] = (
            chunk["sentiment"]
            .apply(get_sentiment_score)
        )

        # Stance
        chunk["main_stance"] = (
            chunk["stance"]
            .apply(get_main_stance)
        )

        chunk["stance_score"] = (
            chunk["stance"]
            .apply(get_stance_score)
        )

        # Remove nested columns
        chunk.drop(
            columns=[
                "stanza_output",
                "stance",
                "sentiment"
            ],
            errors="ignore",
            inplace=True
        )

        # Save as JSONL
        chunk_json = chunk.to_json(
            orient="records",
            lines=True,
            force_ascii=False
        )

        with open(
            OUTPUT_FILE,
            "w" if first_chunk else "a",
            encoding="utf-8"
        ) as f:
            f.write(chunk_json)

        print(f"Processed records up to line {line_no}")

        first_chunk = False
        records = []

# Process final incomplete chunk
if records:

    chunk = pd.DataFrame(records)

    chunk.drop(
        columns=[
            "tweet_id",
            "verified",
            "stanza_named_entities",
            "follower_count",
            "channel",
            "image_tags"
        ],
        errors="ignore",
        inplace=True
    )

    chunk["reconstructed_text"] = (
        chunk["stanza_output"]
        .apply(reconstruct_text)
    )

    chunk["main_sentiment"] = (
        chunk["sentiment"]
        .apply(get_main_sentiment)
    )

    chunk["sentiment_score"] = (
        chunk["sentiment"]
        .apply(get_sentiment_score)
    )

    chunk["main_stance"] = (
        chunk["stance"]
        .apply(get_main_stance)
    )

    chunk["stance_score"] = (
        chunk["stance"]
        .apply(get_stance_score)
    )

    chunk.drop(
        columns=[
            "stanza_output",
            "stance",
            "sentiment"
        ],
        errors="ignore",
        inplace=True
    )

    chunk_json = chunk.to_json(
        orient="records",
        lines=True,
        force_ascii=False
    )

    with open(
        OUTPUT_FILE,
        "a",
        encoding="utf-8"
    ) as f:
        f.write(chunk_json)

    print("Final chunk processed and saved")

print("Dataset processing complete.")

Processed records up to line 10000
Processed records up to line 20000
Processed records up to line 30000
Processed records up to line 40000
Processed records up to line 50000
Processed records up to line 60000
Processed records up to line 70000
Processed records up to line 80000
Processed records up to line 90000
Processed records up to line 100000
Processed records up to line 110000
Processed records up to line 120000
Processed records up to line 130000
Processed records up to line 140000
Processed records up to line 150000
Processed records up to line 160000
Processed records up to line 170000
Processed records up to line 180000
Processed records up to line 190000
Processed records up to line 200000
Processed records up to line 210000
Processed records up to line 220000
Processed records up to line 230000
Processed records up to line 240000
Processed records up to line 250000
Processed records up to line 260000
Processed records up to line 270000
Processed records up to line 280000
P

In [ ]:
import pandas as pd

df = pd.read_json(
    "/content/processed_dataset.jsonl",
    lines=True
)

print(df.shape)

(1524822, 13)


In [ ]:
df.columns

Index(['lang', 'country', 'reconstructed_text', 'main_sentiment',
       'sentiment_score', 'main_stance', 'stance_score', 'question_1',
       'question_2', 'question_3', 'reply_couneng_Latnt',
       'reply_coueng_Latnnt', 'ita_text'],
      dtype='object')

In [ ]:
df['question_1'].value_counts()

,count
question_1,
0.0,831883
1.0,428004


In [ ]:
df['question_2'].value_counts()

,count
question_2,
1.0,950523
0.0,309364


In [ ]:
df['question_3'].value_counts()

,count
question_3,
1.0,1259887


In [ ]:
df['reply_coueng_Latnnt'].value_counts()

,count
reply_coueng_Latnnt,
1.0,1


In [ ]:
df['reply_couneng_Latnt'].value_counts()

,count
reply_couneng_Latnt,
0.0,2


In [ ]:
df['ita_text'].value_counts()

,count
ita_text,
Donbass: the great offensive,1


In [ ]:
required_cols = [
    "lang",
    "country",
    "reconstructed_text",
    "main_sentiment",
    "sentiment_score",
    "main_stance",
    "stance_score"
]

df = df[required_cols]

In [ ]:
df.to_json(
    "/content/processed_dataset_clean.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

In [ ]:
import pandas as pd
df = pd.read_json(
    "/content/processed_dataset_clean.jsonl",
    lines=True
)

print(df.shape)
df.head()

(1524822, 7)


,lang,country,reconstructed_text,main_sentiment,sentiment_score,main_stance,stance_score
0,cs,Slovakia,Weekend selection: Zelensky was not prepared f...,neutral,0.5044,Unsure,0.8667
1,ro,Italy,"The Ukrainian war, Charles Michel of Kiev, the...",neutral,0.9146,Unsure,0.4485
2,te,India,The invention of the Shark drone is a new chap...,neutral,0.5799,Unsure,0.8056
3,te,India,Will nuclear war be over? Those countries that...,negative,0.7900,Unsure,0.2205
4,te,India,Boys fight: Students fight in coaching center ...,negative,0.6543,Pro Ukraine,0.9286


In [ ]:
df.duplicated().sum()

np.int64(65669)

In [ ]:
df = df.drop_duplicates()

In [ ]:
print(df.shape)

(1459153, 7)


In [ ]:
print(df.duplicated().sum())

0


In [ ]:
df.to_csv(
    "/content/processed_dataset_final.csv",
    index=False,
    encoding="utf-8"
)

In [ ]:
df = pd.read_csv(
    "/content/processed_dataset_final.csv"
)

print(df.shape)
df.head()

(1459153, 7)


,lang,country,reconstructed_text,main_sentiment,sentiment_score,main_stance,stance_score
0,cs,Slovakia,Weekend selection: Zelensky was not prepared f...,neutral,0.5044,Unsure,0.8667
1,ro,Italy,"The Ukrainian war, Charles Michel of Kiev, the...",neutral,0.9146,Unsure,0.4485
2,te,India,The invention of the Shark drone is a new chap...,neutral,0.5799,Unsure,0.8056
3,te,India,Will nuclear war be over? Those countries that...,negative,0.7900,Unsure,0.2205
4,te,India,Boys fight: Students fight in coaching center ...,negative,0.6543,Pro Ukraine,0.9286


In [ ]:
df.isnull().sum()

,0
lang,0
country,0
reconstructed_text,24
main_sentiment,0
sentiment_score,0
main_stance,0
stance_score,0


In [ ]:
df.duplicated().sum()

np.int64(0)